# Geolocation Automated Ingestion — Production

## Objective
Incrementally ingest new Geolocation CSV files from the S3 raw landing
folder into the existing Bronze Geolocation Delta table.

Bronze remains raw and preserves the source representation. All business
transformations, casting, cleaning, and ZIP-level aggregation remain in
the existing Silver Geolocation notebook.

---
## Source
`s3://olist-retail-project/raw/geolocation/`

## Checkpoint
`s3://olist-retail-project/_checkpoints/geolocation_ingestion/`

## Target
`workspace.bronze.geolocation`

## Schema contract
- geolocation_zip_code_prefix STRING
- geolocation_lat STRING
- geolocation_lng STRING
- geolocation_city STRING
- geolocation_state STRING

## Important
This notebook does NOT overwrite the Bronze table. Auto Loader processes
only files that have not already been processed by this checkpoint.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType

SOURCE_PATH = "s3://olist-retail-project/raw/geolocation/"
CHECKPOINT_PATH = "s3://olist-retail-project/_checkpoints/geolocation_ingestion/"
SCHEMA_PATH = "s3://olist-retail-project/_schemas/geolocation_ingestion/"
BRONZE_TABLE = "workspace.bronze.geolocation"

EXPECTED_COLUMNS = [
    "geolocation_zip_code_prefix",
    "geolocation_lat",
    "geolocation_lng",
    "geolocation_city",
    "geolocation_state",
]

EXPECTED_SCHEMA = StructType([
    StructField("geolocation_zip_code_prefix", StringType(), True),
    StructField("geolocation_lat", StringType(), True),
    StructField("geolocation_lng", StringType(), True),
    StructField("geolocation_city", StringType(), True),
    StructField("geolocation_state", StringType(), True),
])

In [0]:
# Create the Bronze schema only if it does not already exist.
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

# Verify the existing Bronze target before starting ingestion.
if not spark.catalog.tableExists(BRONZE_TABLE):
    raise ValueError(
        f"Bronze target does not exist: {BRONZE_TABLE}. "
        "Create the existing Bronze table before starting automated ingestion."
    )

before_count = spark.table(BRONZE_TABLE).count()
print(f"Current Bronze Geolocation rows : {before_count:,}")
print(f"Bronze target                    : {BRONZE_TABLE}")

Current Bronze Geolocation rows : 1,000,163
Bronze target                    : workspace.bronze.geolocation


In [0]:
# Verify the existing Bronze schema contract before writing new data.
bronze_schema = spark.table(BRONZE_TABLE).schema
bronze_columns = spark.table(BRONZE_TABLE).columns

if bronze_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze Geolocation schema validation failed.\n"
        f"Expected columns: {EXPECTED_COLUMNS}\n"
        f"Actual columns  : {bronze_columns}"
    )

for field in bronze_schema.fields:
    if field.name in EXPECTED_COLUMNS and field.dataType.simpleString() != "string":
        raise ValueError(
            f"Bronze schema validation failed: "
            f"{field.name} must be STRING but is {field.dataType.simpleString()}."
        )

print("PASS — Existing Bronze Geolocation schema matches the contract.")

PASS — Existing Bronze Geolocation schema matches the contract.


In [0]:
# Auto Loader incrementally discovers new CSV files in the raw/geolocation folder.
# The explicit schema keeps Bronze aligned with the existing raw-preservation design.
geolocation_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH)
        .option("cloudFiles.schemaEvolutionMode", "failOnNewColumns")
        .schema(EXPECTED_SCHEMA)
        .load(SOURCE_PATH)
)

print("PASS — Auto Loader stream configured for new Geolocation files.")
print(f"Source     : {SOURCE_PATH}")
print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Target     : {BRONZE_TABLE}")
print("Schema mode: Strict")

PASS — Auto Loader stream configured for new Geolocation files.
Source     : s3://olist-retail-project/raw/geolocation/
Checkpoint : s3://olist-retail-project/_checkpoints/geolocation_ingestion/
Target     : workspace.bronze.geolocation
Schema mode: Strict


In [0]:
# Append only newly discovered files into the existing Bronze Delta table.
query = (
    geolocation_stream.writeStream
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .toTable(BRONZE_TABLE)
)

query.awaitTermination()

print("PASS — Geolocation incremental ingestion completed successfully.")

PASS — Geolocation incremental ingestion completed successfully.


In [0]:
# Re-read Bronze after ingestion and verify that the table is available.
bronze_after_df = spark.table(BRONZE_TABLE)
after_count = bronze_after_df.count()

print(f"Current Bronze Geolocation rows : {after_count:,}")
print(f"Bronze target                   : {BRONZE_TABLE}")

if after_count < before_count:
    raise ValueError(
        "Bronze population validation failed: "
        "row count decreased after incremental ingestion."
    )

print("PASS — Bronze Geolocation table is available after ingestion.")

Current Bronze Geolocation rows : 2,000,326
Bronze target                   : workspace.bronze.geolocation
PASS — Bronze Geolocation table is available after ingestion.


In [0]:
# Confirm the Bronze schema did not change after ingestion.
final_columns = bronze_after_df.columns
final_schema = bronze_after_df.schema

if final_columns != EXPECTED_COLUMNS:
    raise ValueError(
        "Bronze Geolocation schema changed unexpectedly.\n"
        f"Expected columns: {EXPECTED_COLUMNS}\n"
        f"Current columns : {final_columns}"
    )

for field in final_schema.fields:
    if field.name in EXPECTED_COLUMNS and field.dataType.simpleString() != "string":
        raise ValueError(
            f"Bronze schema changed unexpectedly: "
            f"{field.name} is {field.dataType.simpleString()}, expected string."
        )

print("PASS — Bronze Geolocation schema remains unchanged.")
print("Current Bronze columns:")
bronze_after_df.printSchema()

PASS — Bronze Geolocation schema remains unchanged.
Current Bronze columns:
root
 |-- geolocation_zip_code_prefix: string (nullable = true)
 |-- geolocation_lat: string (nullable = true)
 |-- geolocation_lng: string (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)



In [0]:
# Bronze is intentionally not deduplicated because the source contains
# multiple records per ZIP prefix and Silver performs the aggregation.
null_key_count = (
    bronze_after_df
        .filter(F.col("geolocation_zip_code_prefix").isNull())
        .count()
)

null_lat_count = (
    bronze_after_df
        .filter(F.col("geolocation_lat").isNull())
        .count()
)

null_lng_count = (
    bronze_after_df
        .filter(F.col("geolocation_lng").isNull())
        .count()
)

print(f"NULL ZIP prefix values : {null_key_count:,}")
print(f"NULL latitude values   : {null_lat_count:,}")
print(f"NULL longitude values  : {null_lng_count:,}")

# Do not reject the source solely because Bronze contains nulls:
# the existing Silver notebook is responsible for filtering invalid
# ZIP/coordinate records before aggregation.
print(
    "PASS — Bronze raw-preservation rule confirmed; "
    "no Bronze deduplication or business filtering was applied."
)

NULL ZIP prefix values : 0
NULL latitude values   : 0
NULL longitude values  : 0
PASS — Bronze raw-preservation rule confirmed; no Bronze deduplication or business filtering was applied.


In [0]:
print("""
===============================================================
GEOLOCATION AUTOMATED INGESTION — SUCCESS
===============================================================
Source       : s3://olist-retail-project/raw/geolocation/
Target       : workspace.bronze.geolocation
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict
Bronze role  : Raw source preservation

Next downstream task:
    SILVER_GEOLOCATION

Silver is responsible for:
    - casting ZIP/coordinates
    - filtering invalid ZIP/coordinates
    - trimming/lowercasing city/state
    - aggregating by ZIP prefix
    - validating ZIP uniqueness
===============================================================
""")


GEOLOCATION AUTOMATED INGESTION — SUCCESS
Source       : s3://olist-retail-project/raw/geolocation/
Target       : workspace.bronze.geolocation
Mode         : Incremental append
File handling: Auto Loader
Schema mode  : Strict
Bronze role  : Raw source preservation

Next downstream task:
    SILVER_GEOLOCATION

Silver is responsible for:
    - casting ZIP/coordinates
    - filtering invalid ZIP/coordinates
    - trimming/lowercasing city/state
    - aggregating by ZIP prefix
    - validating ZIP uniqueness



In [0]:
# %sql
# DESCRIBE HISTORY workspace.bronze.geolocation;


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-08-12T09:30:56.000Z,74950116427109,harshrajparmar0117@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> a227e3b5-0856-4bf6-a991-c5498c4933fe, epochId -> 0, statsOnLoad -> true)",null,List(546810179112358),c4b9e16e-930a-44a3-b020-0a4d155bfb13,0812-090159-b75qfeen-v2n,3,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 1000163, numOutputBytes -> 15699138, numAddedFiles -> 1)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
3,2026-08-11T09:27:15.000Z,74950116427109,harshrajparmar0117@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6fbac6b7-710c-48ae-9e41-be51e509d957"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dd01781c-4830-488e-9f16-1f153459ee55""}, statsOnLoad -> true)","List(372327894559921, CUSTOMER360_END_TO_END_PIPELINE, 1122407542353949, 992684920135459, 74950116427109, manual, 7474648188402610)",List(954525762634554),8b18dab2-918e-4ec3-8555-63da78351815,0811-092638-jqtigxms-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000163, numOutputBytes -> 16127687)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
2,2026-08-11T09:27:07.000Z,74950116427109,harshrajparmar0117@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6fbac6b7-710c-48ae-9e41-be51e509d957"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dd01781c-4830-488e-9f16-1f153459ee55""}, statsOnLoad -> false)","List(372327894559921, CUSTOMER360_END_TO_END_PIPELINE, 1122407542353949, 992684920135459, 74950116427109, manual, 7474648188402610)",List(954525762634554),81475f83-1457-4152-a5fb-af391d073f2b,0811-092638-jqtigxms-v2n,1,WriteSerializable,false,Map(),null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
1,2026-08-08T12:26:47.000Z,74950116427109,harshrajparmar0117@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6fbac6b7-710c-48ae-9e41-be51e509d957"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dd01781c-4830-488e-9f16-1f153459ee55""}, statsOnLoad -> true)",null,List(954525762634554),6af169c0-df3f-4ea1-8a7e-2e531585a0f5,0808-103648-iwlyxwj5-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000163, numOutputBytes -> 16127635)",null,Databricks-Runtime/18.x-aarch64-photon-scala2.13
0,2026-08-08T12:26:33.000Z,74950116427109,harshrajparmar0117@gmail.com,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.p

In [0]:
# %sql
# RESTORE TABLE workspace.bronze.geolocation
# TO VERSION AS OF 3;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
16127687,1,1,0,15699138,0


In [0]:
# %sql
# SELECT
#     COUNT(*) AS total_rows,
#     COUNT(geolocation_zip_code_prefix) AS zip_rows,
#     COUNT(geolocation_lat) AS lat_rows,
#     COUNT(geolocation_lng) AS lng_rows
# FROM workspace.bronze.geolocation;

total_rows,zip_rows,lat_rows,lng_rows
1000163,1000163,1000163,1000163


In [0]:
# from pyspark.sql import functions as F

# BRONZE = "workspace.bronze.geolocation"
# SILVER = "workspace.silver.geolocation"

# # The new test ZIP prefixes
# TEST_ZIPS = ["99981", "99982"]

# bronze = spark.table(BRONZE)
# silver = spark.table(SILVER)

# print(f"Current Bronze Geolocation rows : {bronze.count():,}")
# print(f"Current Silver Geolocation rows : {silver.count():,}")

# # ---------------------------------------------------------------
# # Bronze: should contain all 3 raw test records
# # ---------------------------------------------------------------

# bronze_test = (
#     bronze
#     .filter(F.col("geolocation_zip_code_prefix").isin(TEST_ZIPS))
# )

# print(f"\nBronze test rows : {bronze_test.count()}")

# display(
#     bronze_test
#     .orderBy("geolocation_zip_code_prefix")
# )

# # ---------------------------------------------------------------
# # Silver: should contain 2 aggregated ZIP-level records
# # ---------------------------------------------------------------

# silver_test = (
#     silver
#     .filter(
#         F.col("geolocation_zip_code_prefix")
#         .cast("string")
#         .isin(TEST_ZIPS)
#     )
# )

# print(f"Silver test rows : {silver_test.count()}")

# display(
#     silver_test
#     .orderBy("geolocation_zip_code_prefix")
# )

Current Bronze Geolocation rows : 1,000,163
Current Silver Geolocation rows : 19,015

Bronze test rows : 0


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state


Silver test rows : 0


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,silver_load_timestamp


In [0]:
# # ================================================================
# # GEOLOCATION INGESTION DIAGNOSTIC — DELTA HISTORY
# # ================================================================

# BRONZE_TABLE = "workspace.bronze.geolocation"

# history = (
#     spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE}")
#     .select(
#         "version",
#         "timestamp",
#         "operation",
#         "operationParameters",
#         "operationMetrics",
#         "job"
#     )
#     .orderBy(F.col("version").desc())
# )

# display(history.limit(10))

version,timestamp,operation,operationParameters,operationMetrics,job
9,2026-08-12T09:54:37.000Z,RESTORE,"Map(version -> 3, timestamp -> null)","Map(numRestoredFiles -> 0, removedFilesSize -> 1989, numRemovedFiles -> 1, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 16127687)","List(654397208971685, Olist_Customer360_Geolocation_Automated, 1078958951847226, 1119579639396297, 74950116427109, manual, 7474648188402610)"
8,2026-08-12T09:54:25.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> a227e3b5-0856-4bf6-a991-c5498c4933fe, epochId -> 2, statsOnLoad -> true)","Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 1989, numAddedFiles -> 1)","List(654397208971685, Olist_Customer360_Geolocation_Automated, 1078958951847226, 1119579639396297, 74950116427109, manual, 7474648188402610)"
7,2026-08-12T09:44:37.000Z,RESTORE,"Map(version -> 3, timestamp -> null)","Map(numRestoredFiles -> 0, removedFilesSize -> 1989, numRemovedFiles -> 1, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 16127687)","List(654397208971685, Olist_Customer360_Geolocation_Automated, 126553641457106, 925933335088626, 74950116427109, manual, 7474648188402610)"
6,2026-08-12T09:44:25.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> a227e3b5-0856-4bf6-a991-c5498c4933fe, epochId -> 1, statsOnLoad -> true)","Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 1989, numAddedFiles -> 1)","List(654397208971685, Olist_Customer360_Geolocation_Automated, 126553641457106, 925933335088626, 74950116427109, manual, 7474648188402610)"
5,2026-08-12T09:39:10.000Z,RESTORE,"Map(version -> 3, timestamp -> null)","Map(numRestoredFiles -> 0, removedFilesSize -> 15699138, numRemovedFiles -> 1, restoredFilesSize -> 0, numDeletionVectorsAdded -> 0, numDeletionVectorsRemoved -> 0, numOfFilesAfterRestore -> 1, tableSizeAfterRestore -> 16127687)",null
4,2026-08-12T09:30:56.000Z,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> a227e3b5-0856-4bf6-a991-c5498c4933fe, epochId -> 0, statsOnLoad -> true)","Map(numRemovedFiles -> 0, numOutputRows -> 1000163, numOutputBytes -> 15699138, numAddedFiles -> 1)",null
3,2026-08-11T09:27:15.000Z,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6fbac6b7-710c-48ae-9e41-be51e509d957"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dd01781c-4830-488e-9f16-1f153459ee55""}, statsOnLoad -> true)","Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000163, numOutputBytes -> 16127687)","List(372327894559921, CUSTOMER360_END_TO_END_PIPELINE, 1122407542353949, 992684920135459, 74950116427109, manual, 7474648188402610)"
2,2026-08-11T09:27:07.000Z,CREATE OR REPLACE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-6fbac6b7-710c-48ae-9e41-be51e509d957"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-dd01781c-4830-488e-9f16-1f153459ee55""}, statsOnLoad -> false)",Map(),"List(372327894559921, CUSTOMER360_END_TO_END_PIPELINE, 1122407542353949, 992684920135459, 74950116427109, manual, 7474648188402610)"
1,2026-08-08T12:26:47.000Z,CREATE OR REPLACE TAB

In [0]:
# # ================================================================
# # CHECKPOINT CONTENTS
# # ================================================================

# CHECKPOINT = "s3://olist-retail-project/_checkpoints/geolocation_ingestion/"

# display(
#     dbutils.fs.ls(CHECKPOINT)
# )

path,name,size,modificationTime
s3://olist-retail-project/_checkpoints/geolocation_ingestion/metadata,metadata,45,1786527049000
s3://olist-retail-project/_checkpoints/geolocation_ingestion/commits/,commits/,0,1786528684506
s3://olist-retail-project/_checkpoints/geolocation_ingestion/offsets/,offsets/,0,1786528684506
s3://olist-retail-project/_checkpoints/geolocation_ingestion/sources/,sources/,0,1786528684506


In [0]:
# # ================================================================
# # TEST FILE METADATA
# # ================================================================

# TEST_FILE = (
#     "s3://olist-retail-project/raw/geolocation/"
#     "geolocation_automation_e2e_20260812_1525.csv"
# )

# display(
#     dbutils.fs.ls(
#         "s3://olist-retail-project/raw/geolocation/"
#     )
# )

path,name,size,modificationTime
s3://olist-retail-project/raw/geolocation/geolocation_automation_e2e_20260812_1525.csv,geolocation_automation_e2e_20260812_1525.csv,213,1786528435000
s3://olist-retail-project/raw/geolocation/geolocation_automation_test_001.csv,geolocation_automation_test_001.csv,213,1786527795000
s3://olist-retail-project/raw/geolocation/olist_geolocation_dataset.csv,olist_geolocation_dataset.csv,61273883,1786444334000


In [0]:
# # ================================================================
# # RESTORE BRONZE GEOLOCATION TO LATEST SUCCESSFUL INGESTION
# # ================================================================

# BRONZE_TABLE = "workspace.bronze.geolocation"

# spark.sql(f"""
# RESTORE TABLE {BRONZE_TABLE} TO VERSION AS OF 8
# """)

# print("PASS — Bronze Geolocation restored to version 8.")

PASS — Bronze Geolocation restored to version 8.


In [0]:
# ================================================================
# VERIFY BRONZE AFTER RESTORE
# ================================================================

bronze = spark.table("workspace.bronze.geolocation")

print(
    f"Current Bronze Geolocation rows : "
    f"{bronze.count():,}"
)

display(
    bronze
    .filter(
        F.col("geolocation_zip_code_prefix")
        .isin("99981", "99982")
    )
    .orderBy("geolocation_zip_code_prefix")
)

Current Bronze Geolocation rows : 1,000,166


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
99981,-23.55052,-46.63331,sao paulo,SP
99981,-23.55100,-46.63400,sao paulo,SP
99982,-25.42840,-49.27330,curitiba,PR


In [0]:
# ================================================================
# CHECK SILVER TEST RECORDS
# ================================================================

SILVER_TABLE = "workspace.silver.geolocation"

silver = spark.table(SILVER_TABLE)

print(
    f"Current Silver Geolocation rows : "
    f"{silver.count():,}"
)

display(
    silver
    .filter(
        F.col("geolocation_zip_code_prefix")
        .cast("string")
        .isin("99981", "99982")
    )
    .orderBy("geolocation_zip_code_prefix")
)

Current Silver Geolocation rows : 19,015


geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,silver_load_timestamp
